# Dispersion
**Topic:** Descriptive Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Calculate** variance and standard deviation from a dataset
- **Explain** why squaring the deviations matters when computing variance
- **Interpret** what a larger standard deviation means about the spread of values around the mean

> **Tip:** Start by moving the **spread slider**, watch the data points fan out and see how the standard deviation value grows while the mean stays fixed.

---
## How we got here

In *04: Central Tendency* we learned to describe where the center of a dataset is. But two datasets can share the exact same mean while looking completely different, one tightly clustered, one wildly spread out. This notebook measures that spread, introducing the second fundamental property of any distribution.

---
## Why this matters for data science

Standard deviation is everywhere in ML: it's used to normalize features before training (so no single feature dominates), to set confidence intervals around predictions, and to measure the volatility of a time series. In finance, standard deviation *is* risk, a stock's volatility is literally its return's standard deviation.

---
## Try it yourself

In [ ]:
from plotly.subplots import make_subplots
from tkh_utils import PALETTE, FONT, base_layout
from tkh_utils import make_output, make_dropdown, display_widget, register_observer
from ipywidgets import VBox
from IPython.display import display, clear_output
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
N       = 30
_data   = 50.0 + np.random.normal(0, 10, N)
_jitter = np.random.uniform(-0.25, 0.25, N)
_X_RANGE = [-10, 110]

# Pre-compute stats once — data is static
_mean   = float(np.mean(_data))
_std    = float(np.std(_data, ddof=1))
_var    = float(np.var(_data, ddof=1))
_q1     = float(np.percentile(_data, 25))
_med    = float(np.median(_data))
_q3     = float(np.percentile(_data, 75))
_iqr    = _q3 - _q1
_min    = float(np.min(_data))
_max    = float(np.max(_data))
_rng    = _max - _min
_sum_sq = float(np.sum((_data - _mean) ** 2))

out = make_output()
measure_dropdown = make_dropdown(
    ["Overview", "Range", "IQR", "Variance", "Standard Deviation"],
    "Overview",
    "Measure:",
)

def render(change=None):
    measure = measure_dropdown.value

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.72, 0.28],
        vertical_spacing=0.04,
    )

    # ── Base: dot jitter + mean line (always present) ─────────────────────────
    dot_opacity = 0.30 if measure == "Variance" else 0.65
    fig.add_trace(go.Scatter(
        x=_data, y=_jitter, mode="markers",
        marker=dict(color=PALETTE["primary"], size=10, opacity=dot_opacity,
                    line=dict(color="white", width=0.5)),
        name="Data points",
        hovertemplate="Value: %{x:.1f}<extra></extra>",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=[_mean, _mean], y=[-0.45, 0.45], mode="lines",
        line=dict(color=PALETTE["secondary"], width=2.5, dash="dash"),
        name=f"Mean = {_mean:.1f}",
    ), row=1, col=1)

    # ── Measure-specific overlay + formula ────────────────────────────────────
    if measure == "Standard Deviation":
        TICK = 0.12
        fig.add_trace(go.Scatter(
            x=[_mean-_std, _mean+_std, None,
               _mean-_std, _mean-_std, None,
               _mean+_std, _mean+_std],
            y=[0, 0, None, -TICK, TICK, None, -TICK, TICK],
            mode="lines",
            line=dict(color=PALETTE["accent"], width=2.5),
            name=f"±1 std dev  ({_mean-_std:.1f} – {_mean+_std:.1f})",
        ), row=1, col=1)
        formula = (
            f"<b>Standard Deviation (sample)</b><br><br>"
            f"s = √[ Σ(xᵢ − x̄)² / (N − 1) ]<br><br>"
            f"  = √( {_sum_sq:.2f} / {N-1} )<br><br>"
            f"  = √{_var:.4f}<br><br>"
            f"  = <b>{_std:.4f}</b><br><br>"
            f"x̄ = {_mean:.2f},   N = {N}"
        )

    elif measure == "Variance":
        dev_x, dev_y = [], []
        for xi, yi in zip(_data, _jitter):
            dev_x += [xi, _mean, None]
            dev_y += [yi, yi,    None]
        fig.add_trace(go.Scatter(
            x=dev_x, y=dev_y, mode="lines",
            line=dict(color=PALETTE["accent"], width=1.2),
            name="Deviations  xᵢ − x̄",
            hoverinfo="skip",
        ), row=1, col=1)
        formula = (
            f"<b>Variance (sample)</b><br><br>"
            f"s² = Σ(xᵢ − x̄)² / (N − 1)<br><br>"
            f"   = {_sum_sq:.2f} / {N-1}<br><br>"
            f"   = <b>{_var:.4f}</b><br><br>"
            f"x̄ = {_mean:.2f},   N = {N}<br>"
            f"Σ(xᵢ − x̄)² = {_sum_sq:.2f}"
        )

    elif measure == "Range":
        idx_min = int(np.argmin(_data))
        idx_max = int(np.argmax(_data))
        fig.add_trace(go.Scatter(
            x=[_min, _max],
            y=[_jitter[idx_min], _jitter[idx_max]],
            mode="markers",
            marker=dict(color=PALETTE["accent"], size=14,
                        line=dict(color="white", width=1.5)),
            text=[f"Min: {_min:.2f}", f"Max: {_max:.2f}"],
            hovertemplate="%{text}<extra></extra>",
            name=f"Min = {_min:.1f}   Max = {_max:.1f}",
        ), row=1, col=1)
        T = 0.08
        fig.add_trace(go.Scatter(
            x=[_min, _max, None, _min, _min, None, _max, _max],
            y=[-0.40, -0.40, None, -0.40-T, -0.40+T, None, -0.40-T, -0.40+T],
            mode="lines",
            line=dict(color=PALETTE["accent"], width=2),
            showlegend=False, hoverinfo="skip",
        ), row=1, col=1)
        formula = (
            f"<b>Range</b><br><br>"
            f"Range = max − min<br><br>"
            f"      = {_max:.2f} − {_min:.2f}<br><br>"
            f"      = <b>{_rng:.2f}</b>"
        )

    elif measure == "IQR":
        fig.add_trace(go.Scatter(
            x=[_q1, _q3, _q3, _q1, _q1],
            y=[-0.45, -0.45, 0.45, 0.45, -0.45],
            fill="toself", fillcolor="rgba(150,180,240,0.18)",
            line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ), row=1, col=1)
        for val, lbl, dash in [
            (_q1,  f"Q1 = {_q1:.1f}",      "dash"),
            (_med, f"Median = {_med:.1f}", "solid"),
            (_q3,  f"Q3 = {_q3:.1f}",      "dash"),
        ]:
            fig.add_trace(go.Scatter(
                x=[val, val], y=[-0.45, 0.45], mode="lines",
                line=dict(color=PALETTE["accent"], width=2, dash=dash),
                name=lbl,
            ), row=1, col=1)
        formula = (
            f"<b>IQR  (Interquartile Range)</b><br><br>"
            f"IQR = Q3 − Q1<br><br>"
            f"    = {_q3:.2f} − {_q1:.2f}<br><br>"
            f"    = <b>{_iqr:.2f}</b><br><br>"
            f"Q1 (25th pct) = {_q1:.2f}<br>"
            f"Median         = {_med:.2f}<br>"
            f"Q3 (75th pct) = {_q3:.2f}"
        )

    else:  # Overview
        formula = (
            f"<b>Dispersion Summary</b><br><br>"
            f"Range         = {_rng:.2f}<br>"
            f"IQR             = {_iqr:.2f}<br>"
            f"Std Dev      = {_std:.2f}<br>"
            f"Variance     = {_var:.2f}"
        )

    # ── Boxplot (always) ──────────────────────────────────────────────────────
    fig.add_trace(go.Box(
        x=_data, orientation="h",
        marker_color=PALETTE["primary"],
        boxmean=True,
        showlegend=False, name="",
        hoverinfo="x",
    ), row=2, col=1)

    # ── Formula box (upper right) ─────────────────────────────────────────────
    annotations = [dict(
        x=0.99, y=0.97, xref="paper", yref="paper",
        text=formula,
        showarrow=False,
        font=dict(size=12, family=FONT["family"]),
        align="left",
        xanchor="right", yanchor="top",
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor=PALETTE["muted"],
        borderwidth=1,
        borderpad=8,
    )]

    layout = base_layout(
        title=f"Dispersion — {measure}",
        xaxis_title="",
        yaxis_title="",
    )
    layout.update(
        height=500,
        showlegend=True,
        legend=dict(x=0.01, y=0.97, xanchor="left", yanchor="top",
                    bgcolor="rgba(0,0,0,0)"),
        annotations=annotations,
        xaxis=dict(range=_X_RANGE),
        xaxis2=dict(range=_X_RANGE, title="Value"),
        yaxis=dict(range=[-0.55, 0.55], showticklabels=False,
                   showgrid=False, zeroline=False),
        yaxis2=dict(showticklabels=False, showgrid=False, zeroline=False),
    )

    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))  # FigureWidget avoids fig.show() side-channel output

register_observer([measure_dropdown], render)
display_widget(VBox([measure_dropdown]), out)
render()


---
## What's happening?

Dispersion measures how far the data points are from the center. The most common measures start by computing each observation's **deviation** from the mean, how far it sits from x̄.

| Measure | Symbol | What it controls |
|---------|--------|-----------------|
| Variance | s² (sample), σ² (population) | Average squared distance from the mean |
| Standard deviation | s or σ | Square root of variance: in original units |
| Range | max − min | Distance from smallest to largest value |
| IQR | Q3 − Q1 | Spread of the middle 50%: outlier-resistant |

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2 \qquad s = \sqrt{s^2}$$

### Why we square the deviations

Raw deviations (x_i − x̄) always sum to zero, positive and negative deviations cancel out. Squaring before averaging fixes this. Taking the square root at the end brings the units back (e.g., dollars instead of dollars²).

Go back to the widget and compare the variance and standard deviation values as you move the spread slider, variance grows faster because squaring amplifies larger deviations disproportionately.

---
## Real-world example: Daily stock returns

A stock with high return volatility has a large standard deviation, its daily returns swing widely around the mean. A stable bond fund has a small standard deviation, returns cluster tightly.

The chart below shows simulated daily returns for a volatile tech stock and a stable bond fund with the same mean return. Notice:

- **Notice:** Both series have approximately the same mean daily return (~0.04%), but the tech stock's values swing between −4% and +4% while the bond fund stays within ±0.5%
- **Notice:** The standard deviation of the tech stock is roughly 8× larger, this is quantified risk
- **Notice:** The bond fund's distribution is taller and narrower; the tech stock's is shorter and wider, same area, different shape

> **Discussion question:** Two portfolios both average 7% annual return. Portfolio A has σ = 12% and Portfolio B has σ = 3%. Under what circumstances would you prefer the higher-risk option?

In [3]:
np.random.seed(55)

# ── Realistic return parameters from historical market data ─────────────────────
# Tech stock: ~daily vol of 2%, Bond fund: ~daily vol of 0.25%
n = 252   # one trading year

tech_returns  = np.random.normal(loc=0.0004, scale=0.020, size=n)
bond_returns  = np.random.normal(loc=0.0004, scale=0.0025, size=n)
days = np.arange(n)

fig = go.Figure()
for returns, name, color in [
    (tech_returns, f"Tech Stock  σ={np.std(tech_returns)*100:.2f}%", PALETTE["primary"]),
    (bond_returns, f"Bond Fund  σ={np.std(bond_returns)*100:.2f}%",  PALETTE["secondary"]),
]:
    fig.add_trace(go.Scatter(
        x=days, y=returns * 100,
        mode="lines",
        line=dict(color=color, width=1.2),
        name=name,
        opacity=0.85,
    ))

layout = base_layout(
    title="Daily Returns: High vs Low Volatility (simulated, 252 trading days)",
    xaxis_title="Trading Day",
    yaxis_title="Daily Return (%)",
)
fig.update_layout(layout)
fig.show()

### Common dispersion measures and when to use each

| Measure | Sensitive to outliers? | Best used when |
|---------|----------------------|----------------|
| Standard deviation | Yes | Data is roughly symmetric, no extreme outliers |
| Variance | Yes | Used in model math (MSE, PCA, ANOVA) |
| IQR (Q3 − Q1) | No | Data is skewed or has outliers |
| Range (max − min) | Very much so | Quick first look only |
| Coefficient of variation (σ/μ) | Moderate | Comparing spread across variables with different scales |

---
## Key takeaway

> **Standard deviation measures the typical distance from the mean; a larger standard deviation means more spread — and in many domains, more spread means more risk or more uncertainty.**

---
*Next up: Shape of Distributions — beyond center and spread, what does the overall shape of the data look like?*